# F02-P2 People

**Site Characterization: People, components 6**

This notebook covers People Demography and People Vulnerability. Population counts are resampled with `sum`; vulnerability classes use `nearest`. Household estimation is excluded.

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.** Complete as scoped: 6.1 People Demography and 6.2 Vulnerability Assessment.

## Handoff

This notebook reads no previous stage. It uses the AOI and the seven People raster inputs, then writes `outputs/<aoi_id>__F02-P2-people.json`. The stage contains components 6.1 and 6.2 for later consumers.


## Setup


In [1]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass

import geopandas as gpd
import numpy as np
import rasterio

from config import *
from common import *

In [2]:
aoi_id = AOI_ID
aoi = prepare_aoi(gpd.read_file(AOI_PATH))
print(f"AOI {aoi_id}: {fmt_ha(aoi.area_ha)}")

results: dict[str, ComponentResult] = {}


AOI aoi1: 2,057 ha


## Population and People Vulnerability

- Component 6.1 reports total population, available sex totals, and the age-group table.
- Component 6.2 reports four independent vulnerability cards: physical, environmental, economic, and social. It does not calculate a composite vulnerability score.
- The notebook saves both components in the `F02-P2-people` stage handoff.


---
## 6.1 People Demography


### Population data

- `gridded_population_v3.tif` provides total estimated population counts per cell.
- `female_pop_v3.tif` and `male_pop_v3.tif` provide female and male population counts in 20 age bands for 2025. The analysis groups these bands into 14 display ranges: 0-4, five-year ranges from 5-9 through 60-64, and 65+.
- Population is a count, so the analysis uses `sum` resampling when it aligns the rasters to the equal-area analysis grid. It sums counts directly and does not multiply them by cell area.
- Sex and age results are shown only when the sex-age rasters fully cover valid total-population cells. Incomplete coverage remains unavailable; the notebook does not estimate households.

CAUTION!!
`gridded_population_v3.tif` and the sex-age rasters are different WorldPop files. Their native grids
are not identical: the total raster's origin is offset from the female/male rasters by under a
millionth of a degree and it is one pixel smaller in each dimension. The female and male rasters
share one grid between them.

| File | Bands | Width x Height | Resolution (deg) | Origin x, y |
|---|---|---|---|---|
| `gridded_population_v3.tif` | 1 | 58620 x 47468 | 0.00083333333 | 92.171665578, 28.548333555 |
| `female_pop_v3.tif` | 20 | 58621 x 47469 | 0.00083333333 | 92.171666298, 28.548333219 |
| `male_pop_v3.tif` | 20 | 58621 x 47469 | 0.00083333333 | 92.171666298, 28.548333219 |


In [3]:
AGE_CODES = ("00", "01", "05", "10", "15", "20", "25", "30", "35", "40",
             "45", "50", "55", "60", "65", "70", "75", "80", "85", "90")


def _check_population_band_contract(path: str, prefix: str) -> None:
    expected = tuple(f"{prefix}_{age}_2025" for age in AGE_CODES)
    with rasterio.open(path) as src:
        if src.count != len(expected) or tuple(src.descriptions) != expected:
            raise ValueError(
                f"{path} must contain these 20 bands in order: {', '.join(expected)}."
            )


def _load_population_bands(path: str, aoi: AOI, like: RasterSlice) -> list[RasterSlice]:
    return [
        load_raster_clipped(path, aoi, resampling="sum", band=band, like=like)
        for band in range(1, 21)
    ]


def _band_total(bands: list[np.ma.MaskedArray], positions: tuple) -> float:
    return sum(float(bands[position - 1].filled(0.0).sum()) for position in positions)


def _has_complete_sex_age_coverage(total_mask: np.ndarray, sex_age_masks: list[np.ndarray]) -> bool:
    return bool(sex_age_masks) and all(np.all(~mask[~total_mask]) for mask in sex_age_masks)


_demo_bands = [np.ma.array([[float(index)]]) for index in range(1, 21)]
assert _band_total(_demo_bands, (1, 2)) == 3.0
assert _band_total(_demo_bands, (15, 16, 17, 18, 19, 20)) == 105.0
_demo_total_mask = np.array([[False, False]])
_demo_sex_age_masks = [np.array([[False, False]]) for _ in range(40)]
assert _has_complete_sex_age_coverage(_demo_total_mask, _demo_sex_age_masks)
_demo_sex_age_masks[-1] = np.array([[False, True]])
assert not _has_complete_sex_age_coverage(_demo_total_mask, _demo_sex_age_masks)


@dataclass(frozen=True)
class AgeGroup:
    age_group: str
    male_population: float | None
    female_population: float | None
    male_percent: float | None
    female_percent: float | None


def analyze_people_demography(aoi: AOI) -> ComponentResult:
    flags: list[str] = []

    total = load_raster_clipped(POP_TOTAL_RASTER, aoi, resampling="sum")
    if total.valid_count == 0:
        return not_applicable(
            "6.1 People Demography",
            "No population data is available for this project area.",
        )

    _check_population_band_contract(POP_FEMALE_RASTER, "f")
    _check_population_band_contract(POP_MALE_RASTER, "m")
    female_bands = _load_population_bands(POP_FEMALE_RASTER, aoi, total)
    male_bands = _load_population_bands(POP_MALE_RASTER, aoi, total)

    female_values = [band.values for band in female_bands]
    male_values = [band.values for band in male_bands]
    total_mask = np.ma.getmaskarray(total.values)
    female_mask = np.logical_or.reduce([np.ma.getmaskarray(band.values) for band in female_bands])
    male_mask = np.logical_or.reduce([np.ma.getmaskarray(band.values) for band in male_bands])
    shared = ~(total_mask | female_mask | male_mask)
    sex_age_complete = _has_complete_sex_age_coverage(
        total_mask,
        [np.ma.getmaskarray(band.values) for band in female_bands + male_bands],
    )
    total_population = float(total.values.filled(0.0).sum())
    age_rows = [
        AgeGroup(
            age_group=label,
            male_population=(m := _band_total(male_values, positions)) if sex_age_complete else None,
            male_percent=safe_pct(m, total_population) if sex_age_complete else None,
            female_population=(f := _band_total(female_values, positions)) if sex_age_complete else None,
            female_percent=safe_pct(f, total_population) if sex_age_complete else None,
        )
        for label, positions in PEOPLE_AGE_GROUPS.items()
    ]
    female_population = (
        sum(row.female_population for row in age_rows if row.female_population is not None)
        if sex_age_complete else None
    )
    male_population = (
        sum(row.male_population for row in age_rows if row.male_population is not None)
        if sex_age_complete else None
    )

    if female_population is not None and male_population is not None:
        sex_total = female_population + male_population
        male_pct = safe_pct(male_population, sex_total)
        female_pct = safe_pct(female_population, sex_total)
    else:
        male_pct = None
        female_pct = None
        flags.append(
            "6.1: sex-age rasters do not completely cover all valid total-population cells; "
            "sex and age breakdown is unavailable."
        )

    if shared.any():
        female_grid = sum(band.values.filled(0.0) for band in female_bands)
        male_grid = sum(band.values.filled(0.0) for band in male_bands)
        shared_sex_total = float((female_grid[shared] + male_grid[shared]).sum())
        shared_population_total = float(total.values.filled(0.0)[shared].sum())
        # Source grids differ by <1e-6 deg; sum resampling of edge cells drifts ~0.02
        # people on this AOI. 1 person guards genuine data errors, not resampling noise.
        if not np.isclose(shared_sex_total, shared_population_total, rtol=1e-6, atol=1.0):
            flags.append(
                "6.1: the sex-age population total does not match the total population "
                "raster on their shared valid coverage."
            )
    else:
        flags.append(
            "6.1: the sex-age rasters have no shared valid coverage with the total population raster."
        )

    if female_population is not None and male_population is not None:
        narrative = (
            f"Based on gridded world population data, the selected area has an estimated "
            f"total population of {total_population:,.0f}, consisting of "
            f"{male_population:,.0f} males and {female_population:,.0f} females."
        )
    else:
        narrative = (
            f"Based on gridded world population data, the selected area has an estimated "
            f"total population of {total_population:,.0f}. The sex and age breakdown is unavailable."
        )

    return ComponentResult(
        component="6.1 People Demography",
        applicable=True,
        narrative=narrative,
        tables={"age_groups": age_rows},
        values={
            "total_population": total_population,
            "male_population": male_population,
            "female_population": female_population,
            "male_pct": male_pct,
            "female_pct": female_pct,
            "chart_series": "age_groups",
            "chart_unit": "people",
            "chart_axis_label": "Estimated population",
        },
        flags=flags,
        rasters={"6.1_population_total": total},
    )


results["6.1"] = analyze_people_demography(aoi)
show_result(results["6.1"])


[6.1 People Demography]
  Based on gridded world population data, the selected area has an estimated total population of 649, consisting of 335 males and 314 females.
  age_groups:


,age_group,male_population,female_population,male_percent,female_percent
0,0-4,26.792112,26.197670,4.127503,4.035926
1,5-9,32.698724,31.138142,5.037456,4.797038
2,10-14,33.273349,31.711589,5.125981,4.885382
3,15-19,31.902352,29.179859,4.914770,4.495352
4,20-24,26.730118,25.267070,4.117953,3.892560
5,25-29,25.154233,23.734248,3.875177,3.656419
6,30-34,24.620490,23.509371,3.792951,3.621775
7,35-39,25.550405,24.125756,3.936210,3.716734
8,40-44,25.403691,23.633447,3.913608,3.640890
9,45-49,23.350844,20.646269,3.597353,3.180695


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\6.1_age_groups.csv


{'total_population': 649.1118032003552,
 'male_population': 334.9552658127669,
 'female_population': 314.1774984996845,
 'male_pct': 51.60042509447893,
 'female_pct': 48.39957490552107,
 'chart_series': 'age_groups',
 'chart_unit': 'people',
 'chart_axis_label': 'Estimated population'}

  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\6.1_population_total.tif


---
## 6.2 Vulnerability Assessment


### Vulnerability data

- `vulnerability_physical_v3.tif` describes physical vulnerability, including exposed infrastructure, buildings, and critical facilities.
- `vulnerability_natural_v3.tif` describes environmental vulnerability, including ecosystem degradation, natural buffers, and ecological resilience.
- `vulnerability_economic_v3.tif` describes economic vulnerability, including climate-sensitive livelihoods, income stability, and diversification.
- `vulnerability_social_v3.tif` describes social vulnerability, including the capacity of people and communities to prepare, respond, and recover.
- These are categorical classes from 1 (Very Low) to 5 (Very High). The analysis uses `nearest` resampling to retain class values, calculates an area-weighted mean class for each dimension, and rounds it to one displayed class. A dimension with no valid coverage is reported as No data.

In [4]:
@dataclass(frozen=True)
class VulnerabilityCard:
    dimension: str
    description: str
    level_code: int | None
    level_label: str
    mean_class: float | None
    assessed_ha: float
    coverage_pct: float
    narrative: str


def _display_vulnerability_score(mean_class: float) -> int:
    return int(np.clip(np.floor(mean_class + 0.5), 1, 5))


assert _display_vulnerability_score(2.49) == 2
assert _display_vulnerability_score(2.50) == 3
assert _display_vulnerability_score(4.60) == 5


VULNERABILITY_COPY = {
    "physical": {
        "description": "Condition of infrastructure, buildings, and critical facilities exposed to hazards.",
        "template": (
            "The selected area demonstrates a {level_label} level of physical vulnerability to "
            "climate hazards. This assessment is based on the condition of infrastructure, "
            "buildings, critical facilities, and other physical assets exposed to climate-related "
            "hazards, which influence the area's ability to withstand and recover from climate impacts."
        ),
    },
    "environmental": {
        "description": "Ecosystem degradation, loss of natural buffers, and ecological resilience.",
        "template": (
            "The environmental conditions in the selected area are classified as having a "
            "{level_label} level of vulnerability to climate hazards. This assessment is based on "
            "the degradation of ecosystems, the loss of natural buffers, and the reduction in "
            "ecological resilience, which influence the ability of ecosystems to withstand and "
            "recover from climate-related impacts."
        ),
    },
    "economic": {
        "description": "Dependence on climate-sensitive livelihoods, income stability, and diversification.",
        "template": (
            "The selected area has a {level_label} level of economic vulnerability to climate "
            "hazards. This assessment is based on the community's dependence on climate-sensitive "
            "livelihoods, household income stability, asset bases, and economic diversification."
        ),
    },
    "social": {
        "description": "Demographic and socio-economic capacity to prepare, respond, and recover.",
        "template": (
            "Communities in the selected area have a {level_label} level of social vulnerability "
            "to climate hazards. This assessment is based on demographic and socio-economic "
            "characteristics that influence the ability of people and communities to prepare for, "
            "respond to, and recover from climate-related impacts."
        ),
    },
}


def analyze_vulnerability_assessment(aoi: AOI) -> ComponentResult:
    cards: list[VulnerabilityCard] = []
    rasters: dict[str, RasterSlice] = {}
    flags: list[str] = []
    scores: dict[str, int | None] = {}
    mean_classes: dict[str, float | None] = {}
    assessed_ha: dict[str, float] = {}
    coverage_pct: dict[str, float] = {}

    for dimension, path in PEOPLE_VULNERABILITY_RASTERS.items():
        description = VULNERABILITY_COPY[dimension]["description"]
        template = VULNERABILITY_COPY[dimension]["template"]
        raster = load_raster_clipped(path, aoi, resampling="nearest")
        valid = raster.values.compressed()
        if valid.size == 0:
            cards.append(VulnerabilityCard(dimension, description, None, "No data", None, 0.0, 0.0, f"No {dimension} vulnerability data is available for this project area."))
            scores[dimension] = mean_classes[dimension] = None
            assessed_ha[dimension] = coverage_pct[dimension] = 0.0
            continue

        if not np.all(np.isclose(valid, np.rint(valid))):
            raise ValueError(f"{dimension} vulnerability contains non-integer class values.")
        codes = np.rint(valid).astype(int)
        unexpected = sorted(set(codes) - set(PEOPLE_VULNERABILITY_LEVELS))
        if unexpected:
            raise ValueError(f"{dimension} vulnerability has invalid class codes: {unexpected}.")

        mean_class = float(valid.sum() * raster.pixel_area_ha / raster.valid_area_ha)
        level_code = _display_vulnerability_score(mean_class)
        level_label = PEOPLE_VULNERABILITY_LEVELS[level_code]
        dimension_assessed_ha = raster.valid_area_ha
        dimension_coverage_pct = safe_pct(dimension_assessed_ha, aoi.area_ha)
        cards.append(VulnerabilityCard(dimension, description, level_code, level_label, mean_class, dimension_assessed_ha, dimension_coverage_pct, template.format(level_label=level_label.lower())))
        scores[dimension] = level_code
        mean_classes[dimension] = mean_class
        assessed_ha[dimension] = dimension_assessed_ha
        coverage_pct[dimension] = dimension_coverage_pct
        rasters[f"6.2_vulnerability_{dimension}"] = raster

    narrative = "\n\n".join(f"{card.dimension.title()}: {card.narrative}" for card in cards)
    return ComponentResult(
        component="6.2 Vulnerability Assessment",
        applicable=any(card.level_code is not None for card in cards),
        narrative=narrative,
        tables={"vulnerability_cards": cards},
        values={"vulnerability_scores": scores, "vulnerability_mean_classes": mean_classes, "assessed_ha": assessed_ha, "coverage_pct": coverage_pct},
        flags=flags,
        rasters=rasters,
    )


This writes `<AOI_ID>__F02-P2-people.json`.


In [5]:
results["6.2"] = analyze_vulnerability_assessment(aoi)
show_result(results["6.2"])

[6.2 Vulnerability Assessment]
  Physical: The selected area demonstrates a low level of physical vulnerability to climate hazards. This assessment is based on the condition of infrastructure, buildings, critical facilities, and other physical assets exposed to climate-related hazards, which influence the area's ability to withstand and recover from climate impacts.

Environmental: The environmental conditions in the selected area are classified as having a high level of vulnerability to climate hazards. This assessment is based on the degradation of ecosystems, the loss of natural buffers, and the reduction in ecological resilience, which influence the ability of ecosystems to withstand and recover from climate-related impacts.

Economic: No economic vulnerability data is available for this project area.

Social: No social vulnerability data is available for this project area.
  vulnerability_cards:


,dimension,description,level_code,level_label,mean_class,assessed_ha,coverage_pct,narrative
0,physical,"Condition of infrastructure, buildings, and cr...",2.0,Low,1.679487,13.633489,0.662721,The selected area demonstrates a low level of ...
1,environmental,"Ecosystem degradation, loss of natural buffers...",4.0,High,4.279367,1734.948828,84.335545,The environmental conditions in the selected a...
2,economic,"Dependence on climate-sensitive livelihoods, i...",NaN,No data,NaN,0.000000,0.000000,No economic vulnerability data is available fo...
3,social,Demographic and socio-economic capacity to pre...,NaN,No data,NaN,0.000000,0.000000,No social vulnerability data is available for ...


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\6.2_vulnerability_cards.csv


{'vulnerability_scores': {'physical': 2,
  'environmental': 4,
  'economic': None,
  'social': None},
 'vulnerability_mean_classes': {'physical': 1.6794871794871795,
  'environmental': 4.279367318154342,
  'economic': None,
  'social': None},
 'assessed_ha': {'physical': 13.633488676351446,
  'environmental': 1734.9488282239033,
  'economic': 0.0,
  'social': 0.0},
 'coverage_pct': {'physical': 0.6627213918498848,
  'environmental': 84.33554532694816,
  'economic': 0.0,
  'social': 0.0}}

  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\6.2_vulnerability_physical.tif
  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\6.2_vulnerability_environmental.tif


---
## Save

Each section above already ran and displayed itself. This cell writes them to one combined JSON.

In [6]:
path = save_results(results, aoi, aoi_id, STAGE_PEOPLE)
print(f"Saved {path}")

Saved D:\NBSTOOLV3\OUTPUTS\aoi1\aoi1__F02-P2-people.json
